In [ ]:
import pydeck as pdk, pandas as pd, numpy as np, seaborn as sns, matplotlib.pyplot as plt, matplotlib.colors as mcolors
from datetime import datetime, timedelta
sns.set_theme(style="ticks")
plt.rcParams['mathtext.fontset'] = 'custom'
plt.rcParams['mathtext.rm'] = 'Arial'
plt.rcParams['mathtext.it'] = 'Arial:italic'

# parameters

In [ ]:
yeari, yearf = '2024', '2024'
weeki, weekf = '18', '31'

In [ ]:
di = datetime.strptime(f'{yeari}-{weeki}-1', "%Y-%W-%w").date()
df = datetime.strptime(f'{yearf}-{weekf}-1', "%Y-%W-%w").date() + timedelta(6)
ds = [di+timedelta(dt) for dt in range((df-di).days+1)]
daylist = ds
print(di, 'until', df)

# load data

In [ ]:
city = 'Stuttgart'
day = '2024-07-05'#'2024-07-05'#'2024-06-28'

In [ ]:
city = 'Düsseldorf'
day = '2024-06-01'#'2024-06-01'#'2024-05-25'

In [ ]:
city = 'Hamburg'
day = '2024-05-11'#'2024-05-11'#'2024-05-04'

In [ ]:
city = 'Köln'
day = '2024-07-21'#'2024-07-21'#'2024-07-14'

In [ ]:
city = 'Gelsenkirchen'
day = '2024-07-27'#'2024-07-27'#'2024-07-20'

In [ ]:
viewstate = {'Stuttgart': [48.78422258241185, 9.18497909163697+.015, 13, 12],
             'Düsseldorf': [51.224949154286364, 6.780327227212697, 13, 12],
             'Hamburg': [53.54490763468703+.005, 9.983793898076929, 13, 12],
             'Köln': [50.93714925074693, 6.952636468657181, 13.5, 14.25],
             'Gelsenkirchen': [51.55447099891889, 7.070060061802468, 12, 12]#[51.55351109326721, 7.064741712211087, 12, 12]
            }
viewstate_pitch = {'Stuttgart': [48.78422258241185-.005, 9.18497909163697+.005, 13, 13],
                   'Düsseldorf': [51.224949154286364-.005, 6.780327227212697, 13, 14.25],
                   'Hamburg': [53.54490763468703+.005, 9.983793898076929, 13, 14.25],
                   'Köln': [50.93714925074693-.0025, 6.952636468657181, 14, 14.25],
                   'Gelsenkirchen': [51.55447099891889, 7.070060061802468, 13, 14.25]#[51.55351109326721-.03, 7.064741712211087+.03, 12, 14.25]
                  }

In [ ]:
data = pd.read_csv(f'data/figs1/map_pings_{city.lower()}_{str(day)}.csv')
data['day'] = [d.date() for d in pd.to_datetime(data.day)]
data['stime'] = pd.to_datetime(data.stime)
data['size'] = 10
# Generate 24 distinct HUSL colors
colors = sns.husl_palette(24)
#colors = colors[12:] + colors[:12]
hour2color = {h: [255*n for n in c]+[160.] for h, c in enumerate(colors)}
data['color'] = data.stime.apply(lambda x: x.hour).map(hour2color)
data

# analyses

In [ ]:
scatter = pdk.Layer(
    "ScatterplotLayer",
    data[['lon','lat','size','color']].to_dict('records'),
    get_position=["lon", "lat"],  # Must match column names
    get_radius="size",
    get_fill_color="color",#[255, 0, 0, 160],
    pickable=True
)

view_state = pdk.ViewState(
    #longitude=data.lon.mean(), latitude=data.lat.mean(),
    longitude=viewstate_pitch[city][1], latitude=viewstate_pitch[city][0],
    zoom=viewstate_pitch[city][2], pitch=45
)

r = pdk.Deck(layers=[scatter], initial_view_state=view_state, map_style="https://basemaps.cartocdn.com/gl/dark-matter-gl-style/style.json")#.show()
# carto light: "https://basemaps.cartocdn.com/gl/positron-gl-style/style.json"
# carto dark: "https://basemaps.cartocdn.com/gl/dark-matter-gl-style/style.json"
# carto voyager: "https://basemaps.cartocdn.com/gl/voyager-gl-style/style.json"
# osm: "https://tiles.stadiamaps.com/styles/osm_bright.json"
city_string = city.lower().replace('ä','ae').replace('ö','oe').replace('ü','ue')
r.to_html(f'plots/figs1/map_pings_{city_string}_{str(day)}.html')

In [ ]:
sns.set_theme(style="ticks")

# Create a colormap and norm
cmap = mcolors.ListedColormap(colors)
bounds = np.arange(25)  # 24 bins = 25 boundaries
norm = mcolors.BoundaryNorm(boundaries=bounds, ncolors=24)

# Create a dummy scalar mappable for the colorbar
fig, ax = plt.subplots(figsize=(6, .5))
fig.subplots_adjust(bottom=0.5)

cb = fig.colorbar(
    mappable=plt.cm.ScalarMappable(cmap=cmap, norm=norm),
    cax=ax,
    orientation='horizontal',
    ticks=np.arange(0.5, 24.5, 1),  # center ticks
    label='hour',
)

# Set tick labels
cb.ax.set_xticks(np.arange(0.5, 24.5, 1))
cb.ax.set_xticklabels([str(i) for i in range(0, 24)])
cb.ax.minorticks_off()

# Remove the black frame
for spine in cb.ax.spines.values():
    spine.set_visible(False)

plt.savefig(f'plots/figs1/map_pings_colorbar.jpg', bbox_inches='tight', dpi=300)
plt.savefig(f'plots/figs1/map_pings_colorbar.pdf', bbox_inches='tight')
plt.show()